In [1]:
import pandas as pd
import math
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Model


In [2]:
df1 = pd.read_csv('datasets/linkedin_jobs_20260418_215155.csv')
df2 = pd.read_csv('datasets/linkedin_jobs_20260416_223428.csv')
df3 = pd.read_csv('datasets/jobstreet_jobs_20260420_205621.csv')
df = pd.concat([df1, df2, df3])
rows, cols = df.shape

print(f"Rows : {rows}")
print(f"Cols : {cols}")
df.info()

Rows : 1895
Cols : 12
<class 'pandas.DataFrame'>
Index: 1895 entries, 0 to 63
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1895 non-null   int64 
 1   title             1895 non-null   str   
 2   company           1894 non-null   str   
 3   location          1831 non-null   object
 4   job_description   1895 non-null   str   
 5   job_url           1895 non-null   str   
 6   search_role       1895 non-null   str   
 7   search_location   1895 non-null   str   
 8   extracted_skills  1895 non-null   str   
 9   skills_count      1895 non-null   int64 
 10  scraped_at        1895 non-null   str   
 11  salary            29 non-null     str   
dtypes: int64(2), object(1), str(9)
memory usage: 5.8+ MB


In [3]:
df = df.drop_duplicates()
df['company'] = df['company'].fillna(df['company'].mode()[0])
df['location'] = df['location'].fillna(df['location'].mode()[0])
df = df.drop(columns='salary', errors='ignore')

print(f"total duplicated columns : {df.duplicated().sum()}")
print(f"total null column : {df.isnull().sum()}")

total duplicated columns : 0
total null column : id                  0
title               0
company             0
location            0
job_description     0
job_url             0
search_role         0
search_location     0
extracted_skills    0
skills_count        0
scraped_at          0
dtype: int64


In [4]:
df["job_description"] = df["job_description"].fillna("").astype(str)
df["extracted_skills"] = df["extracted_skills"].fillna("").astype(str)

df["text_input"] = df["job_description"] + " " + df["extracted_skills"]

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])

num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")

print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'], 
    df['label'], 
    test_size=0.2, 
    random_state=42,
    stratify=df['label'] 
)

max_vocab_length = 10000 
max_sequence_length = 250

vectorizer = TextVectorization(
    max_tokens=max_vocab_length,
    output_mode='int',
    output_sequence_length=max_sequence_length
)

vectorizer.adapt(X_train.to_numpy())

print("\nShape X_train:", X_train.shape)
print("Contoh kalimat pertama setelah di-vektorisasi (angka):")
print(vectorizer(X_train.iloc[0:1]).numpy())

Total role unik: 3
Mapping kelas:
{'Data Scientist': 0, 'Web Developer': 1, 'web': 2}

Shape X_train: (1516,)
Contoh kalimat pertama setelah di-vektorisasi (angka):
[[  21    4   19   89    5 2464    4  426  116  294 4166 3737  115    5
  1662   61    2  294  768    8 5202 5279  931 3710  198 1093  503    2
   369  184  115 2220   32 4991    7  650    6 4762  295  971   13  419
   538   15   31  935  133    3  167    8  535   92    2  345  426   12
     3   14  295   17    8   88  960   16   25   46    4  122    3  951
     5 1094 2250  114   23  105   16    7    8 2684    6  205  787   12
    16   25  744  679   26   81 2381  124    5    8  430   22   69    2
    70    8 2701   65    6   28  185 1772   90   25   32 2366   20    8
  2300  699   13  745   16    3  158 1342  672  271  425   32  193    3
  1653    3   30  341  238  811 1769  366 1444   91    4 2140   27  172
    13  395   20 8851    2 1481 2690  746    1 1674 2138    3  329    1
   538 2454   13   22  220    7  706    2 2

In [5]:
print(type(X_train), X_train.dtype)
print(X_train.head(3))
print(X_train.astype(str).to_numpy().dtype)

<class 'pandas.Series'> str
578    About the job\n\nResponsibilities\n\nIn partic...
89     About the job\n\nJob Description\n\n1. Identif...
361    About the job\n\nJob Description\n\n\nDrive co...
Name: text_input, dtype: str
object


In [6]:
# 1. Bikin Custom Callback (Syarat Rubrik)
class CustomStopCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Stop training kalau akurasi validasi udah di atas 85%
        if logs.get('val_accuracy') >= 0.85:
            print("\n[INFO] Akurasi validasi sudah mencapai 85%, menghentikan training untuk mencegah overfitting!")
            self.model.stop_training = True

In [ ]:

embedding_dim = 64

inputs = Input(shape=(), dtype=tf.string, name="text_input")
x = vectorizer(inputs)
x = Embedding(input_dim=max_vocab_length, output_dim=embedding_dim, name="embedding_layer")(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.5)(x)
outputs = Dense(num_classes, activation="softmax", name="role_output")(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier")
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

X_train_np = X_train.astype(str).to_numpy(dtype=object)
X_val_np   = X_val.astype(str).to_numpy(dtype=object)

y_train_np = y_train.to_numpy(dtype="int32")
y_val_np   = y_val.to_numpy(dtype="int32")

vectorizer.adapt(X_train_np)

print("Train size:", len(X_train_np))
print("Batch size:", 32)
print("Steps/epoch:", math.ceil(len(X_train_np)/32))
print("GPUs:", tf.config.list_physical_devices("GPU"))

history = model.fit(
    X_train_np, y_train_np,
    validation_data=(X_val_np, y_val_np),
    epochs=15,
    batch_size=32,
    callbacks=[CustomStopCallback()],
    verbose=2
)

Train size: 1516
Batch size: 32
Steps/epoch: 48
GPUs: []
Epoch 1/15
